In [ ]:
'''
USE FOR SPARE PARTS ONLY
'''

"\ntry wavecal at similar airmass as data (same time observation)\n\ndo fxcor on standards at order 6 (atmosphere water feature)\n- standard shouldn't be variable\n- if there is a nonzero rv, then we know the telluric correction is not working (can't use same wavecal for everything)\n- could try applying the difference to the target, may work may not\n\nif wavecal bad, need to get a wavecal for each night\n- for each object, even, could just take same object same image\n- no A-B, just the individual spectra, one aperture\n- no telluric correction\n- extract each spectrum in the ABBA series, using itself as the wavecal (A1 has wavecal A1.wcal)\n    - could also combine all the spectra from that target for that run as wavecals\n- find rv of those 4 images (first image template), so each observation has 4 rvs\n    - beware of telluric (atmosphere) lines; H band is good for avoiding this\n    - alternatively, combine them and get rv without doing telluric correction\n\nrun fxcor on whole s

In [2]:
import os
import sys
from astropy.io import fits
import numpy as np
from collections import defaultdict
import shutil
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time

In [3]:
maindir = 'Main'
workdir = 'Rawtest'
os.chdir(maindir)

observatory = 'apo'
ra_decs = {}
overwrite = True

In [4]:
def make_lists(input_dir):
    """
    Groups FITS images by star name, creates a template file for each star, 
    and writes out cleanly organized text file lists for whole and split bands.
    """
    
    star_groups = defaultdict(list)

    output_dir = input_dir
    for fname in sorted(os.listdir(input_dir)):
        if fname.lower().endswith(".fits") and "template" not in fname.lower():
            sname = fname.split('.')[0]
            star_groups[sname].append(fname)
        
    star_list = list(star_groups.keys())

    for sname, images in star_groups.items():
        print(f'Processing star: {sname}')

        template_name = f'{sname}.template.fits'
        shutil.copyfile(os.path.join(input_dir, images[0]), os.path.join(input_dir, template_name))
        print(f' Created template file: {template_name}')
        ra_decs[sname] = get_sky_coord(sname)
        distribute_files(images, sname, output_dir)

    distribute_files(star_list, 'star', output_dir)
    return star_list

In [5]:
def distribute_files(file_list, list_name, output_dir):
    """
    Writes image lists to file text arrays matching their star names.
    """
    file_path = os.path.join(output_dir, f'{list_name}list.txt')

    with open(file_path, 'w', encoding='utf-8', newline='\n') as f:
        for item in file_list:
            f.write(f'{item.strip()}\n')

In [6]:
def get_obs_epoch(header):
    # Try common epoch keywords
    if 'DATE-OBS' in header:
        return Time(header['DATE-OBS']).jyear
    elif 'MJD-OBS' in header:
        return Time(header['MJD-OBS'], format='mjd').jyear
    elif 'AVE_MJD' in header:
        return Time(header['AVE_MJD'], format='mjd').jyear
    return None

In [7]:
def get_sky_coord(name):
    query = name[:-3] + "_" + name[-3:]
    result = Simbad.query_object(query)

    if result is None:
        raise ValueError(f"No SIMBAD result found for {query}")

    ra_text = str(result["RA"][0])
    dec_text = str(result["DEC"][0])

    # SIMBAD RA is in hours; SIMBAD Dec is in degrees
    coord = SkyCoord(ra=ra_text, dec=dec_text, unit=(u.hourangle, u.deg), frame="fk5", equinox="J2000")
    print(f'Got J2000 coordinates for {name}')
    return coord

In [8]:
def get_hour_angle(coord, time, location='apo'):
    location_coords = (location)
    lst = time.sidereal_time('apparent', longitude=location_coords.lon)

    lha = lst - coord.ra
    lha_wrapped = lha.wrap_at(12*u.hourangle)
    print(lha_wrapped)
    return lha_wrapped

In [9]:
def convert_ra_dec(coord_j2000, starname, obs_epoch=None):
    '''
    Take J2000 skycoords and convert them to the epoch of observation
    '''
    if obs_epoch:
        # Transform from J2000 to observation epoch
        coord_obs = coord_j2000.transform_to(SkyCoord(frame="fk5", equinox=obs_epoch.tt))
    else:
        print(f"{starname} used original coords")
        coord_obs = coord_j2000
        
    ra_formatted = coord_obs.ra.to_string(unit=u.hourangle, sep=":", precision=2, pad=True)
    dec_formatted = coord_obs.dec.to_string(unit=u.deg, sep=":", precision=1, alwayssign=True, pad=True)

    return [ra_formatted, dec_formatted]

In [10]:
def fill_header(infile, fname):
    with fits.open(infile) as hdul:
        hdr = hdul[0].header
        data = hdul[0].data

    # === Get exposure time ===
    t1 = Time(f"{hdr['SRT_DATE']}T{hdr['SRT_TIME']}", format = 'isot', scale = 'utc')
    t2 = Time(f"{hdr.get('END_DATE')}T{hdr.get('END_TIME')}", format = 'isot', scale = 'utc')
    exptime = (t2-t1).sec

    # === Epoch calculation ===
    t_start = Time(float(hdr["SRT_MJD"]), format = 'mjd', scale = 'utc')
    JD = t_start.jd
    epoch = t_start.jyear
    starname = fname.split('.')[0]

    hdr["EXPTIME"] = (exptime, 'Exposure time in seconds') 
    hdr["RA"], hdr["DEC"] = convert_ra_dec(ra_decs[starname], starname)
    # new_ra, new_dec = convert_ra_dec(ra_decs[starname], starname, epoch)
    # hdr["RA_OBS"], hdr["DEC_OBS"] = (new_ra, "Right Ascension, observed epoch"), (new_dec, "Declination, observed epoch")
                    
    # hdr["EPOCH"] = (epoch, 'Year of observation relative to J2000.0')

    # new names for existing values
    hdr["DATE-OBS"] = (f"{hdr.get('SRT_DATE')}", 'Start observation date in UTC')
    hdr["UT"] = (f"{hdr.get('SRT_TIME')}", 'Start observation time in UTC')
    hdr["MJD-OBS"] = (hdr.get("SRT_MJD"), hdr.comments['SRT_MJD'])
    hdr["OBSERVAT"]=(hdr.get("OBSERVAT",observatory))
    hdr["OBJECT"] = (hdr.get("OBJNAME", "Unknown"))
    hdr["FILENAME"] = fname

    hdr.add_history('replaced missing ra, dec, exposure time, observatory values.')

    fits.writeto(infile, data, header=hdr, overwrite=True)

In [11]:
def refactor_all():
    for fname in os.listdir(workdir):
        if fname.lower().endswith(".fits"):
            infile = os.path.join(workdir, fname)
            fbase = fname.replace(".fits","")
            fill_missing_data(infile, 'Apache Point Observatory')

In [12]:
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u
from astroquery.simbad import Simbad
from astropy.io import fits
import numpy as np
import time

def fill_missing_data(fits_file, obs_name):
    print(fits_file)
    hdul = fits.open(fits_file)
    header = hdul[0].header
    
    # Extract values from FITS header
    name = header.get('OBJNAME').strip().replace(" ", "")
    fits_ra = header.get('RA').strip()
    fits_dec = header.get('DEC').strip()
    fits_pa = header.get('PA')
    
    srt_date_str = header.get('SRT_DATE').strip()
    srt_time_str = header.get('SRT_TIME').strip()
    ave_date_str = header.get('AVE_DATE').strip()
    ave_time_str = header.get('AVE_TIME').strip()
    end_date_str = header.get('END_DATE').strip()
    end_time_str = header.get('END_TIME').strip()

    '''
    # SIMBAD lookup
    result = Simbad.query_object(name)
    
    if result is None:
        hdul.close()
        exit(1)
    
    ra_text = str(result["RA"][0])
    dec_text = str(result["DEC"][0])
    star_coord = SkyCoord(ra=ra_text, dec=dec_text, unit=(u.hourangle, u.deg), frame="fk5", equinox="J2000")
    '''
    star_coord = ra_decs[name]
    # Observatory location
    observatory = EarthLocation.of_site(obs_name)
    
    # Create observation times
    srt_time = Time(f"{srt_date_str}T{srt_time_str}", scale='utc')
    ave_time = Time(f"{ave_date_str}T{ave_time_str}", scale='utc')
    end_time = Time(f"{end_date_str}T{end_time_str}", scale='utc')
    
    # Calculate hour angles
    srt_ha = srt_time.sidereal_time('apparent', longitude=observatory.lon) - star_coord.ra
    ave_ha = ave_time.sidereal_time('apparent', longitude=observatory.lon) - star_coord.ra
    end_ha = end_time.sidereal_time('apparent', longitude=observatory.lon) - star_coord.ra
    
    srt_ha = srt_ha.wrap_at(12*u.hourangle)
    ave_ha = ave_ha.wrap_at(12*u.hourangle)
    end_ha = end_ha.wrap_at(12*u.hourangle)
    
    # Calculate position angle (parallactic angle)
    lat = observatory.lat
    ha = ave_ha
    dec = star_coord.dec
    
    pa_parallactic = np.arctan2(
        np.sin(ha),
        np.cos(dec) * np.tan(lat) - np.sin(dec) * np.cos(ha)
    )
    pa_parallactic = u.Quantity(pa_parallactic, u.radian).to(u.degree)
    
    # Update FITS header
    ra_str = star_coord.ra.to_string(unit=u.hourangle, sep=':', precision=2, pad=True)
    dec_str = star_coord.dec.to_string(unit=u.degree, sep=':', precision=1, pad=True)
    
    srt_ha_str = srt_ha.to_string(unit=u.hourangle, sep=':', precision=1, pad=True)
    ave_ha_str = ave_ha.to_string(unit=u.hourangle, sep=':', precision=2, pad=True)
    end_ha_str = end_ha.to_string(unit=u.hourangle, sep=':', precision=1, pad=True)
    
    header['RA'] = (ra_str, 'Right Ascension, FK5 J2000 (from SIMBAD)')
    header['DEC'] = (dec_str, 'Declination, FK5 J2000 (from SIMBAD)')
    header['SRT_HA'] = (srt_ha_str, 'Start hour angle (hours) - calculated')
    header['AVE_HA'] = (ave_ha_str, 'Average hour angle (hours) - calculated')
    header['END_HA'] = (end_ha_str, 'End hour angle (hours) - calculated')
    header['PA'] = (float(pa_parallactic.value), 'Position angle E of N (deg) - parallactic angle')
    header['OBSERVAT'] = (obs_name, 'Observatory')
    
    # Write to file
    output_file = fits_file
    hdul.writeto(output_file, overwrite=True)
    
    hdul.close()

In [13]:
def main():
    print('running')
    star_list = make_lists(workdir)
    refactor_all()
    print('finished!')

In [14]:
if __name__ == '__main__':
    main()

running
Processing star: BMPer
 Created template file: BMPer.template.fits
Got J2000 coordinates for BMPer
Processing star: CPCep
 Created template file: CPCep.template.fits
Got J2000 coordinates for CPCep
Processing star: ERAur
 Created template file: ERAur.template.fits
Got J2000 coordinates for ERAur
Processing star: ETCVn
 Created template file: ETCVn.template.fits
Got J2000 coordinates for ETCVn
Processing star: ETVul
 Created template file: ETVul.template.fits
Got J2000 coordinates for ETVul
Processing star: GYSge
 Created template file: GYSge.template.fits
Got J2000 coordinates for GYSge
Processing star: IYCyg
 Created template file: IYCyg.template.fits
Got J2000 coordinates for IYCyg
Processing star: MNCam
 Created template file: MNCam.template.fits
Got J2000 coordinates for MNCam
Processing star: OTPer
 Created template file: OTPer.template.fits
Got J2000 coordinates for OTPer
Processing star: RSOri
 Created template file: RSOri.template.fits
Got J2000 coordinates for RSOri
Pr